In [0]:

import requests
import json
import os
import re
from datetime import datetime, timezone

# 1. Konfigurierbare Unternehmensliste
dbutils.widgets.text(
    "boards",
    "anthropic,cloudflare,figma"
)

boards = [
    board.strip().lower()
    for board in dbutils.widgets.get("boards").split(",")
    if board.strip()
]

if (
    not boards
    or len(boards) != len(set(boards))
    or any(not re.fullmatch(r"[a-z0-9_-]+", b) for b in boards)
):
    raise ValueError("Invalid or duplicate board tokens")

# 2. Gemeinsame Snapshot-Kennung
now = datetime.now(timezone.utc)

snapshot_id = now.strftime("%Y%m%dT%H%M%S%fZ")
snapshot_at = now.isoformat()

landing_path = (
    "/Volumes/job_market/landing/raw_files/"
    "greenhouse_snapshots/"
)

# 3. Alle Unternehmen zuerst erfolgreich abrufen
snapshots = {}

with requests.Session() as session:

    for board in boards:

        url = (
            "https://boards-api.greenhouse.io"
            f"/v1/boards/{board}/jobs"
        )

        response = session.get(
            url,
            params={"content": "true"},
            timeout=60
        )
        response.raise_for_status()

        data = response.json()

        jobs = data.get("jobs")
        meta = data.get("meta") or {}

        if (
            not isinstance(jobs, list)
            or not jobs
            or meta.get("total") != len(jobs)
        ):
            raise ValueError(
                f"Invalid or incomplete API response: {board}"
            )

        ids = [job.get("id") for job in jobs]

        if None in ids or len(ids) != len(set(ids)):
            raise ValueError(f"Invalid job IDs: {board}")

        snapshots[board] = {
            "source": "greenhouse",
            "board_token": board,
            "snapshot_id": snapshot_id,
            "snapshot_at": snapshot_at,
            "meta": meta,
            "jobs": jobs
        }

        print(f"Fetched {board}: {len(jobs)} jobs")

# 4. Erst nach erfolgreichen Abrufen speichern
os.makedirs(landing_path, exist_ok=True)

for board, snapshot in snapshots.items():

    filename = (
        f"greenhouse_{snapshot_id}_{board}.json"
    )

    file_path = os.path.join(
        landing_path, filename
    )

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False)

    print(f"Saved: {file_path}")

print("Extraction completed.")
print("Snapshot ID:", snapshot_id)
print("Companies:", len(snapshots))
print(
    "Total jobs:",
    sum(len(s["jobs"]) for s in snapshots.values())
)
